# Yelp Data Cleaning — Philadelphia
Filters to Philadelphia restaurants, extracts attributes, computes survival fields, and picks an example restaurant.

In [1]:
import json
import pandas as pd
from collections import defaultdict

print('Libraries loaded.')

Libraries loaded.


## 1. Define chain list and Philadelphia zip codes

In [13]:
KNOWN_CHAINS = {
    'mcdonald', 'subway', 'starbucks', 'wendy', 'wawa', 'chipotle',
    'dunkin', 'kfc', 'burger king', 'taco bell', 'pizza hut', 'domino',
    'chick-fil-a', 'panda express', 'five guys', 'panera', 'sonic',
    'popeyes', 'dairy queen', 'little caesars', 'papa john', 'in-n-out',
    'shake shack', 'whataburger', 'arbys', "arby's", 'hardee', "carl's jr",
    'jack in the box', 'del taco', 'checkers', 'rally', 'wingstop',
    'jersey mike', 'jimmy john', 'firehouse subs', 'potbelly', 'quiznos',
    'baskin robbins', 'cold stone', 'orange julius',
    'ihop', 'denny', 'waffle house', 'cracker barrel', 'olive garden',
    'red lobster', 'applebee', 'chili', 'ruby tuesday', 'outback',
    'longhorn', 'buffalo wild wings', 'hooters', 'texas roadhouse',
    'cheesecake factory', 'p.f. chang', 'benihana', 'raising cane', 'philly pretzel factory'
}

PHILADELPHIA_ZIPS = {
    '19101', '19102', '19103', '19104', '19105', '19106', '19107', '19108',
    '19109', '19110', '19111', '19112', '19114', '19115', '19116', '19118',
    '19119', '19120', '19121', '19122', '19123', '19124', '19125', '19126',
    '19127', '19128', '19129', '19130', '19131', '19132', '19133', '19134',
    '19135', '19136', '19137', '19138', '19139', '19140', '19141', '19142',
    '19143', '19144', '19145', '19146', '19147', '19148', '19149', '19150',
    '19151', '19152', '19153', '19154', '19160', '19176', '19190', '19192'
}

def is_chain(name):
    return any(chain in name.lower() for chain in KNOWN_CHAINS)

def normalize_zip(value):
    if not value:
        return ''
    return str(value).strip().split('-', 1)[0].zfill(5)

print(f'{len(KNOWN_CHAINS)} chains defined.')
print(f'{len(PHILADELPHIA_ZIPS)} Philadelphia zip codes defined.')

60 chains defined.
56 Philadelphia zip codes defined.


## 2. Load business.json — filter to Philadelphia restaurants

In [14]:
print('Loading business data...')
businesses = []

with open('yelp_academic_dataset_business.json', 'r', encoding='utf-8') as f:
    for line in f:
        b = json.loads(line)

        # Filter: restaurants only
        categories = b.get('categories') or ''
        if 'Restaurant' not in categories and 'Food' not in categories:
            continue

        # Filter: Philadelphia zip codes only
        if normalize_zip(b.get('postal_code')) not in PHILADELPHIA_ZIPS:
            continue

        attrs = b.get('attributes') or {}
        delivery    = attrs.get('RestaurantsDelivery')
        takeout     = attrs.get('RestaurantsTakeOut')
        price_range = attrs.get('RestaurantsPriceRange2')
        parking_raw = attrs.get('BusinessParking')

        has_parking = False
        if isinstance(parking_raw, dict):
            has_parking = any(parking_raw.values())
        elif isinstance(parking_raw, str):
            has_parking = 'True' in parking_raw

        businesses.append({
            'business_id':  b.get('business_id'),
            'name':         b.get('name'),
            'city':         b.get('city'),
            'state':        b.get('state'),
            'postal_code':  normalize_zip(b.get('postal_code')),
            'latitude':     b.get('latitude'),
            'longitude':    b.get('longitude'),
            'stars':        b.get('stars'),
            'review_count': b.get('review_count'),
            'is_open':      b.get('is_open'),
            'categories':   categories,
            'delivery':     delivery,
            'takeout':      takeout,
            'price_range':  price_range,
            'has_parking':  has_parking,
            'is_chain':     is_chain(b.get('name', '')),
        })

df = pd.DataFrame(businesses)
print(f'Philadelphia restaurants found: {len(df)}')
df.head(3)

Loading business data...
Philadelphia restaurants found: 7073


,business_id,name,city,state,postal_code,latitude,longitude,stars,review_count,is_open,categories,delivery,takeout,price_range,has_parking,is_chain
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",False,True,1,True,False
1,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,1,"Sushi Bars, Restaurants, Japanese",True,True,2,True,False
2,ROeacJQwBeh05Rqg7F6TCg,BAP,Philadelphia,PA,19147,39.943223,-75.162568,4.5,205,1,"Korean, Restaurants",None,True,1,True,False


## 3. Load review.json — first and last review date per business

In [15]:
philly_ids = set(df['business_id'])
print(f'Loading reviews for {len(philly_ids)} Philadelphia restaurants...')
print('(This may take a few minutes)')

review_dates = defaultdict(list)

with open('yelp_academic_dataset_review.json', 'r', encoding='utf-8') as f:
    for line in f:
        r = json.loads(line)
        bid  = r.get('business_id')
        date = r.get('date')
        if bid in philly_ids and date:
            review_dates[bid].append(date)

print(f'Reviews loaded for {len(review_dates)} businesses.')

first_review = {bid: sorted(dates)[0]  for bid, dates in review_dates.items()}
last_review  = {bid: sorted(dates)[-1] for bid, dates in review_dates.items()}

df['first_review_date'] = df['business_id'].map(first_review)
df['last_review_date']  = df['business_id'].map(last_review)

df['first_review_date'] = pd.to_datetime(df['first_review_date'], errors='coerce')
df['last_review_date']  = pd.to_datetime(df['last_review_date'],  errors='coerce')

print('Dates merged.')
df[['name', 'first_review_date', 'last_review_date']].head(3)

Loading reviews for 7073 Philadelphia restaurants...
(This may take a few minutes)
Reviews loaded for 7073 businesses.
Dates merged.


,name,first_review_date,last_review_date
0,St Honore Pastries,2008-03-09 00:36:56,2021-11-01 18:22:07
1,Tuna Bar,2017-11-25 02:26:49,2022-01-14 00:35:07
2,BAP,2013-12-10 22:13:17,2022-01-14 05:17:18


## 4. Compute survival fields

In [16]:
df['observation_days']  = (df['last_review_date'] - df['first_review_date']).dt.days
df['observation_years'] = (df['observation_days'] / 365).round(1)

dataset_cutoff = df['last_review_date'].max()
print(f'Dataset cutoff date: {dataset_cutoff.date()}')

df['survived'] = (
    (df['is_open'] == 1) &
    ((dataset_cutoff - df['last_review_date']).dt.days < 730)
)

df[['name', 'is_open', 'survived', 'observation_years']].head(5)

Dataset cutoff date: 2022-01-19


,name,is_open,survived,observation_years
0,St Honore Pastries,1,True,13.7
1,Tuna Bar,1,True,4.1
2,BAP,1,True,8.1
3,Bar One,0,False,3.3
4,DeSandro on Main,0,False,5.3


## 5. Drop rows with missing critical fields

In [17]:
before = len(df)
df = df.dropna(subset=['latitude', 'longitude', 'stars', 'first_review_date'])
print(f'Dropped {before - len(df)} rows. Remaining: {len(df)}')

Dropped 0 rows. Remaining: 7073


In [24]:
# Drop non-restaurant businesses
EXCLUDE_NAMES = {'cvs pharmacy', 'gopuff', 'acme markets'}

before = len(df)
df = df[~df['name'].str.lower().isin(EXCLUDE_NAMES)]
df_ind = df[df['is_chain'] == False]
df_chain = df[df['is_chain'] == True]
print(f'Dropped {before - len(df)} non-restaurant rows. Remaining: {len(df)}')

Dropped 0 non-restaurant rows. Remaining: 7025


## 6. Save outputs

In [25]:
df_ind   = df[df['is_chain'] == False]
df_chain = df[df['is_chain'] == True]

df.to_json('restaurants_clean.json',           orient='records', indent=2, date_format='iso')
df_ind.to_json('restaurants_independent.json', orient='records', indent=2, date_format='iso')
df_chain.to_json('restaurants_chain.json',     orient='records', indent=2, date_format='iso')

print(f'restaurants_clean.json           — {len(df)} records')
print(f'restaurants_independent.json     — {len(df_ind)} records')
print(f'restaurants_chain.json           — {len(df_chain)} records')

restaurants_clean.json           — 7025 records
restaurants_independent.json     — 6488 records
restaurants_chain.json           — 537 records


## 7. Pick example restaurant for simulator (Image 5)

In [29]:
candidates = df_ind[
    (df_ind['is_open'] == 0) &
    (df_ind['stars'] >= 3.5) &
    (df_ind['review_count'] >= 50)
].sort_values('review_count', ascending=False)

print(f'{len(candidates)} candidate restaurants found.')
print()

# Show top 5 so you can pick manually if you prefer
print('── Top 5 candidates ──')
print(candidates[['name', 'city', 'stars', 'review_count', 'first_review_date', 'last_review_date']].head(5).to_string())

example = df_ind[df_ind['name'] == "Sabrina's Café"].iloc[0]
print()
print('── Selected example ──')
print(f'  Name:         {example["name"]}')
print(f'  Stars:        {example["stars"]}')
print(f'  Reviews:      {example["review_count"]}')
print(f'  First review: {example["first_review_date"].date()}')
print(f'  Last review:  {example["last_review_date"].date()}')

with open('example_restaurant.json', 'w') as f:
    json.dump(example.to_dict(), f, indent=2, default=str)
print('Saved: example_restaurant.json')

721 candidate restaurants found.

── Top 5 candidates ──
                        name          city  stars  review_count   first_review_date    last_review_date
6673          Federal Donuts  Philadelphia    4.0          1464 2012-10-01 18:16:07 2021-11-15 20:10:02
4980          Sabrina's Café  Philadelphia    4.0          1176 2005-09-11 23:32:56 2021-06-06 00:21:26
409                    Jones  Philadelphia    3.5          1141 2005-12-19 18:51:01 2021-11-12 20:06:07
2931                Farmicia  Philadelphia    4.0          1094 2006-03-06 16:28:26 2020-09-06 23:00:08
3276  Garces Trading Company  Philadelphia    4.0           896 2010-02-16 18:24:56 2018-06-26 14:27:10

── Selected example ──
  Name:         Sabrina's Café
  Stars:        4.0
  Reviews:      1721
  First review: 2007-11-19
  Last review:  2022-01-19
Saved: example_restaurant.json


## 8. Summary stats

In [27]:
print('── Summary ──')
print(f'Total Philadelphia restaurants: {len(df)}')
print(f'Independent:                    {len(df_ind)}')
print(f'Chain:                          {len(df_chain)}')
print(f'Survival rate (independent):    {df_ind["survived"].mean():.1%}')
print(f'Survival rate (chain):          {df_chain["survived"].mean():.1%}')
print(f'Avg stars (independent):        {df_ind["stars"].mean():.2f}')
print(f'Avg stars (chain):              {df_chain["stars"].mean():.2f}')
print(f'Dataset cutoff:                 {dataset_cutoff.date()}')

── Summary ──
Total Philadelphia restaurants: 7025
Independent:                    6488
Chain:                          537
Survival rate (independent):    53.3%
Survival rate (chain):          70.4%
Avg stars (independent):        3.67
Avg stars (chain):              2.45
Dataset cutoff:                 2022-01-19


In [28]:
df_ind.nsmallest(20, 'stars')[['name', 'stars', 'review_count']]

,name,stars,review_count
510,Pizano's Family Pizza and BreakFast Kitchen,1.0,11
566,Ruby's Roof Jamaican Restaurant,1.0,5
600,Paradise Pizzeria,1.0,5
1116,Seafood Sensations,1.0,5
1431,Marabello's,1.0,6
1849,Bocci's Steakhouse & Comedy Cafe,1.0,5
1897,The Playhouse 822,1.0,10
1911,Montego Grill,1.0,5
2595,World Bean,1.0,8
5760,Geno's Steaks,1.0,5


In [2]:
# Compute the city-wide averages needed for the Image 5 report card
import pandas as pd
df = pd.read_json('data/yelp_filtered/restaurants_clean.json')

print("=== Report Card averages ===")

# Rating
print(f"Average rating: {df['stars'].mean():.2f}")
print(f"Independent average rating: {df[df['is_chain']==False]['stars'].mean():.2f}")

# Reviews
print(f"Average review count: {df['review_count'].mean():.0f}")
print(f"Median review count: {df['review_count'].median():.0f}")

# Parking
parking_rate = df['has_parking'].mean()
print(f"Share with parking: {parking_rate:.1%}")

# Delivery
def to_bool(v): return 1 if v in [True, 'True', 'true', 1] else 0
df['_deliv'] = df['delivery'].apply(to_bool)
print(f"Share offering delivery: {df['_deliv'].mean():.1%}")

# Price distribution
print("\nPrice distribution:")
print(df['price_range'].value_counts().sort_index())

=== Report Card averages ===
Average rating: 3.58
Independent average rating: 3.67
Average review count: 102
Median review count: 33
Share with parking: 65.3%
Share offering delivery: 54.5%

Price distribution:
price_range
1       2628
2       2946
3        301
4         42
None       1
Name: count, dtype: int64


In [2]:
import json
from collections import defaultdict

data = json.load(open('data/yelp_filtered/restaurants_clean.json'))
N = len(data)
OVERALL = 100.0 * sum(r['survived'] for r in data) / N

# delivery / takeout are stored as the strings "True" / "False"
def has_delivery(r): return str(r['delivery']) == 'True'
def has_takeout(r):  return str(r['takeout'])  == 'True'

def survival(rows):
    rows = list(rows)
    s = sum(1 for r in rows if r['survived'])
    return s, len(rows), (100.0 * s / len(rows) if rows else 0.0)

# 1) Each factor on its own, to see which ones actually move survival
print('overall: %d/%d = %.1f%%' % survival(data))
print('delivery True : %d/%d = %.1f%%' % survival(r for r in data if has_delivery(r)))
print('delivery False: %d/%d = %.1f%%' % survival(r for r in data if not has_delivery(r)))
print('takeout  True : %d/%d = %.1f%%' % survival(r for r in data if has_takeout(r)))
print('takeout  False: %d/%d = %.1f%%' % survival(r for r in data if not has_takeout(r)))

# 2) Cuisine: survival per category token (tokens with enough rows), then a
#    "favorable cuisine" = the restaurant lists any above-average cuisine
DROP = {'Restaurants', 'Food'}
cui = defaultdict(lambda: [0, 0])
for r in data:
    for t in (x.strip() for x in str(r['categories']).split(',')):
        if t and t not in DROP:
            cui[t][0] += 1
            cui[t][1] += int(bool(r['survived']))
favorable_cuisines = {t for t, (n, s) in cui.items()
                      if n >= 80 and 100.0 * s / n > OVERALL}

def favorable_cuisine(r):
    return any(t.strip() in favorable_cuisines
               for t in str(r['categories']).split(','))

# 3) Count favorable factors per restaurant, then survival by count
def favorable_count(r):
    return int(has_delivery(r)) + int(has_takeout(r)) + int(favorable_cuisine(r))

print('\nfavorable factors vs survival:')
buckets = defaultdict(lambda: [0, 0])
for r in data:
    k = favorable_count(r)
    buckets[k][0] += 1
    buckets[k][1] += int(bool(r['survived']))
for k in sorted(buckets):
    n, s = buckets[k]
    print('  %d favorable -> %d/%d = %.1f%%' % (k, s, n, 100.0 * s / n))

overall: 3836/7025 = 54.6%
delivery True : 2637/3827 = 68.9%
delivery False: 1199/3198 = 37.5%
takeout  True : 3259/5678 = 57.4%
takeout  False: 577/1347 = 42.8%

favorable factors vs survival:
  0 favorable -> 54/210 = 25.7%
  1 favorable -> 532/1392 = 38.2%
  2 favorable -> 1047/2293 = 45.7%
  3 favorable -> 2203/3130 = 70.4%


In [3]:
import json
from datetime import datetime
from collections import defaultdict

# adjust path to wherever the file sits in your repo
data = json.load(open('data/yelp_filtered/restaurants_clean.json'))
OVERALL = 100.0 * sum(r['survived'] for r in data) / len(data)

# delivery / takeout are stored as the strings "True" / "False"
def has_delivery(r): return str(r['delivery']) == 'True'
def has_takeout(r):  return str(r['takeout'])  == 'True'

# favorable category: the restaurant lists any category whose own survival rate
# beats the city average (only categories common enough to trust, n >= 80)
DROP = {'Restaurants', 'Food'}
cui = defaultdict(lambda: [0, 0])
for r in data:
    for t in (x.strip() for x in str(r['categories']).split(',')):
        if t and t not in DROP:
            cui[t][0] += 1
            cui[t][1] += int(bool(r['survived']))
favorable_cats = {t for t, (n, s) in cui.items()
                  if n >= 80 and 100.0 * s / n > OVERALL}

def favorable_category(r):
    return any(t.strip() in favorable_cats
               for t in str(r['categories']).split(','))
def favorable_count(r):
    return int(has_delivery(r)) + int(has_takeout(r)) + int(favorable_category(r))
def lifespan_years(r):
    a = datetime.fromisoformat(r['first_review_date'][:19])
    b = datetime.fromisoformat(r['last_review_date'][:19])
    return (b - a).days / 365.25

# closer candidates: well-loved but closed early, missing the factors that move
# survival -> no delivery, at most one favorable factor, typical-closure lifespan
closers = [r for r in data
           if not r['survived'] and r['stars'] >= 4.3 and not has_delivery(r)
           and favorable_count(r) <= 1 and 2 <= lifespan_years(r) <= 7
           and r['review_count'] >= 120]
closers.sort(key=lambda r: -r['review_count'])

# survivor candidates: still open, all three factors, comparable rating
survivors = [r for r in data
             if r['survived'] and favorable_count(r) == 3
             and 4.0 <= r['stars'] <= 4.5 and r['review_count'] >= 300]
survivors.sort(key=lambda r: -r['review_count'])

def show(r):
    fav_tok = [t.strip() for t in str(r['categories']).split(',')
               if t.strip() in favorable_cats]
    print('  %-22s %.1f*  %4d rev  %.1fy  delivery=%-5s takeout=%-5s favCat=%-5s [%s]  factors=%d/3  %s'
          % (r['name'][:22], r['stars'], r['review_count'], lifespan_years(r),
             has_delivery(r), has_takeout(r), favorable_category(r),
             '/'.join(fav_tok) or '-', favorable_count(r),
             'OPEN' if r['survived'] else 'CLOSED'))

print('closer candidates:')
for r in closers: show(r)
print('\nsurvivor candidates (top 6):')
for r in survivors[:6]: show(r)
print('\nchosen for the report card:')
for name in ["Kanella South", "Barbuzzo"]:
    show(next(r for r in data if r['name'] == name))

closer candidates:
  Khmer Kitchen          4.5*   372 rev  6.7y  delivery=False takeout=True  favCat=False [-]  factors=1/3  CLOSED
  Tredici Enoteca        4.5*   352 rev  5.0y  delivery=False takeout=False favCat=True  [Bars/Nightlife]  factors=1/3  CLOSED
  Kanella South          4.5*   249 rev  2.7y  delivery=False takeout=False favCat=True  [Bars/Nightlife]  factors=1/3  CLOSED

survivor candidates (top 6):
  Reading Terminal Marke 4.5*  5721 rev  16.1y  delivery=True  takeout=True  favCat=True  [Shopping/Fast Food/Beer/Wine & Spirits/Seafood/Bagels/Specialty Food/Juice Bars & Smoothies/Coffee & Tea/Bakeries/Grocery]  factors=3/3  OPEN
  El Vez                 4.0*  3187 rev  16.4y  delivery=True  takeout=True  favCat=True  [Bars/Nightlife/Breakfast & Brunch/Mexican]  factors=3/3  OPEN
  Barbuzzo               4.5*  2893 rev  11.4y  delivery=True  takeout=True  favCat=True  [Pizza]  factors=3/3  OPEN
  Parc                   4.0*  2761 rev  13.5y  delivery=True  takeout=True  fav